# 📉 Notebook 6: Stock Price Simulation via Euler Method & GBM
**FinTech Stock Market Analysis — Optimization Techniques**

This notebook applies the **Euler Method** and **ODE/SDE** theory to simulate future stock price paths.

### Geometric Brownian Motion (GBM)
Stock prices are modelled by the stochastic differential equation (SDE):
$$dS = \mu S \, dt + \sigma S \, dW_t$$
where:
- $S$ = stock price
- $\mu$ = drift (expected daily return)
- $\sigma$ = volatility (standard deviation of daily returns)
- $W_t$ = Wiener process (Brownian motion), $dW_t \sim \mathcal{N}(0, dt)$

### Euler-Maruyama Discretization
The **Euler method for ODEs** applied to this SDE gives the **Euler-Maruyama** scheme:
$$\boxed{S_{t+\Delta t} = S_t + \mu S_t \Delta t + \sigma S_t \sqrt{\Delta t} \cdot Z_t, \quad Z_t \sim \mathcal{N}(0,1)}$$
This is the direct analogue of the classical Euler ODE step $y_{n+1} = y_n + f(t_n, y_n)\,h$.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

TARGET_TICKER = 'JPM'   # change to analyse a different stock

df = pd.read_csv(f'../data/{TARGET_TICKER}_processed.csv',
                 index_col=0, parse_dates=True)
print(f'✅ Loaded {TARGET_TICKER}: {len(df)} trading days')
print(f'   Period: {df.index[0].date()} → {df.index[-1].date()}')
df[['Close', 'Daily_Return']].tail(3)

## Part 1 — Parameter Estimation

We estimate the GBM parameters **$\mu$** (drift) and **$\sigma$** (volatility) from the historical daily return series:
$$\hat{\mu} = \frac{1}{T}\sum_{t=1}^T r_t, \qquad \hat{\sigma} = \sqrt{\frac{1}{T-1}\sum_{t=1}^T (r_t - \hat{\mu})^2}$$

Annualized values (using 252 trading days): $\mu_{ann} = \hat{\mu} \times 252$, $\sigma_{ann} = \hat{\sigma} \times \sqrt{252}$

In [ ]:
daily_ret = df['Daily_Return'].dropna()

mu    = daily_ret.mean()   # daily drift
sigma = daily_ret.std()    # daily volatility
S0    = float(df['Close'].iloc[-1])   # starting price = last known close

mu_ann    = mu    * 252
sigma_ann = sigma * np.sqrt(252)

print('📊 GBM PARAMETER ESTIMATION')
print('='*50)
print(f'  Stock          : {TARGET_TICKER}')
print(f'  Current Price  : ${S0:.2f}')
print(f'  Observations   : {len(daily_ret)} trading days')
print(f'\n  Daily Drift   μ : {mu:.8f}  →  {mu_ann*100:.2f}% annualized')
print(f'  Daily Volatility σ : {sigma:.8f}  →  {sigma_ann*100:.2f}% annualized')

# Visualise the return distribution
fig_ret = go.Figure()
fig_ret.add_trace(go.Histogram(
    x=daily_ret, nbinsx=60,
    marker_color='#00e5ff', opacity=0.75, name='Daily Returns'
))
fig_ret.add_vline(x=mu, line_dash='dash', line_color='#ffd740',
                  annotation_text=f'μ={mu:.5f}', annotation_font_color='#ffd740')
fig_ret.update_layout(
    template='plotly_dark', height=350,
    title=f'{TARGET_TICKER} — Historical Daily Return Distribution',
    xaxis_title='Daily Return', yaxis_title='Frequency'
)
fig_ret.show()

## Part 2 — Euler-Maruyama Implementation

**Analogy with Euler's Method for ODEs:**

| Classical Euler (ODE) | Euler-Maruyama (SDE) |
|---|---|
| $\dot{y} = f(t, y)$ | $dS = \mu S\,dt + \sigma S\,dW$ |
| $y_{n+1} = y_n + f(t_n, y_n)\cdot h$ | $S_{t+dt} = S_t + \mu S_t dt + \sigma S_t \sqrt{dt}\cdot Z$ |
| $h$ = step size | $dt = 1/252$ (1 trading day) |
| Deterministic increment | Random increment $Z \sim \mathcal{N}(0,1)$ |

Running this for many independent random draws $Z_t$ produces a **Monte Carlo ensemble** of possible price paths.

In [ ]:
N_DAYS    = 252       # simulation horizon: 1 year
N_PATHS   = 500       # number of Monte Carlo paths
dt        = 1.0 / 252 # time step in years

np.random.seed(42)

# paths[t, i] = price of path i at day t
paths      = np.zeros((N_DAYS + 1, N_PATHS))
paths[0]   = S0

for t in range(1, N_DAYS + 1):
    Z           = np.random.standard_normal(N_PATHS)    # Z ~ N(0,1)
    # Euler-Maruyama step: S(t+dt) = S(t) + μ·S(t)·dt + σ·S(t)·√dt·Z
    paths[t]    = paths[t-1] + mu * paths[t-1] * dt + sigma * paths[t-1] * np.sqrt(dt) * Z

final_prices = paths[-1]

print(f'✅ Simulated {N_PATHS} paths over {N_DAYS} trading days')
print(f'\nFinal Price Statistics (Day {N_DAYS}):')
print(f'  Mean   : ${final_prices.mean():.2f}')
print(f'  Median : ${np.median(final_prices):.2f}')
print(f'  Std Dev: ${final_prices.std():.2f}')
print(f'  5th %%ile  : ${np.percentile(final_prices,  5):.2f}')
print(f'  95th %%ile : ${np.percentile(final_prices, 95):.2f}')

## Part 3 — Exact GBM Solution vs Euler-Maruyama

GBM has a known **closed-form solution** (via Itô's lemma):
$$S(T) = S_0 \cdot \exp\!\left[\left(\mu - \frac{\sigma^2}{2}\right)T + \sigma W_T\right]$$

The **Itô correction term** $-\sigma^2/2$ arises because Brownian motion is not differentiable — the Euler-Maruyama method approximates this but converges to the exact solution as $\Delta t \to 0$.

In [ ]:
T = N_DAYS * dt   # total time in years

# Theoretical mean and std of the exact GBM distribution
exact_mean = S0 * np.exp(mu_ann * T)
exact_std  = S0 * np.exp(mu_ann * T) * np.sqrt(np.exp(sigma_ann**2 * T) - 1)

euler_mean = final_prices.mean()
euler_std  = final_prices.std()

err_mean = abs(euler_mean - exact_mean) / exact_mean * 100
err_std  = abs(euler_std  - exact_std)  / exact_std  * 100

print('📐 EULER-MARUYAMA vs EXACT GBM SOLUTION')
print('='*55)
print(f'  Exact GBM: S(T) = S₀ · exp((μ − σ²/2)·T + σ·W(T))')
print(f'  Itô correction term: −σ²/2 = {-sigma_ann**2/2:.6f}')
print()
print(f'  Horizon T         : {T:.4f} years  ({N_DAYS} trading days)')
print(f'  Theoretical Mean  : ${exact_mean:.2f}')
print(f'  Simulated Mean    : ${euler_mean:.2f}  (error: {err_mean:.3f}%)')
print(f'  Theoretical Std   : ${exact_std:.2f}')
print(f'  Simulated Std     : ${euler_std:.2f}  (error: {err_std:.3f}%)')
print(f'\n  ✅ Small errors confirm Euler-Maruyama convergence (dt = 1/252 is very fine)')

In [ ]:
# ── Visualisation 1: Simulation Paths with Confidence Bands ──────────────────
t_axis = np.arange(N_DAYS + 1)
p5  = np.percentile(paths,  5, axis=1)
p25 = np.percentile(paths, 25, axis=1)
p50 = np.percentile(paths, 50, axis=1)
p75 = np.percentile(paths, 75, axis=1)
p95 = np.percentile(paths, 95, axis=1)

fig_sim = go.Figure()

# Individual paths (faded)
n_show = min(120, N_PATHS)
for i in range(n_show):
    fig_sim.add_trace(go.Scatter(
        x=t_axis, y=paths[:, i], mode='lines',
        line=dict(color='rgba(0,229,255,0.05)', width=1),
        showlegend=False, hoverinfo='skip'
    ))

# 90% confidence band
fig_sim.add_trace(go.Scatter(
    x=t_axis, y=p95, mode='lines',
    line=dict(color='rgba(255,215,64,0)'), showlegend=False
))
fig_sim.add_trace(go.Scatter(
    x=t_axis, y=p5, mode='lines',
    fill='tonexty', fillcolor='rgba(255,215,64,0.12)',
    line=dict(color='rgba(255,215,64,0)'),
    name='90% Confidence Band'
))

# 50% confidence band
fig_sim.add_trace(go.Scatter(
    x=t_axis, y=p75, mode='lines',
    line=dict(color='rgba(100,255,150,0)'), showlegend=False
))
fig_sim.add_trace(go.Scatter(
    x=t_axis, y=p25, mode='lines',
    fill='tonexty', fillcolor='rgba(100,255,150,0.18)',
    line=dict(color='rgba(100,255,150,0)'),
    name='50% Confidence Band'
))

# Median path
fig_sim.add_trace(go.Scatter(
    x=t_axis, y=p50, mode='lines',
    line=dict(color='#ffd740', width=2.5),
    name='Median Path'
))

# Current price reference
fig_sim.add_hline(y=S0, line_dash='dash', line_color='white', opacity=0.35,
                  annotation_text=f'Today: ${S0:.2f}')

fig_sim.update_layout(
    template='plotly_dark', height=550,
    title=(f'📉 GBM Monte Carlo — {TARGET_TICKER}  '
           f'({N_PATHS} paths, Euler-Maruyama, dt=1/252)'),
    xaxis_title='Trading Days Forward',
    yaxis_title='Simulated Price ($)'
)
fig_sim.show()

In [ ]:
# ── Visualisation 2: Final Price Distribution ─────────────────────────────────
fig_dist = go.Figure()
fig_dist.add_trace(go.Histogram(
    x=final_prices, nbinsx=60,
    marker_color='#00e5ff', opacity=0.7,
    name='Simulated Final Prices'
))
fig_dist.add_vline(x=S0,
    line_dash='dash', line_color='white', opacity=0.5,
    annotation_text=f'Start ${S0:.0f}')
fig_dist.add_vline(x=np.percentile(final_prices, 5),
    line_dash='dot', line_color='#ff5252',
    annotation_text='5th %ile')
fig_dist.add_vline(x=np.percentile(final_prices, 50),
    line_dash='dash', line_color='#ffd740',
    annotation_text='Median')
fig_dist.add_vline(x=np.percentile(final_prices, 95),
    line_dash='dot', line_color='#00e676',
    annotation_text='95th %ile')
fig_dist.update_layout(
    template='plotly_dark', height=380,
    title=f'{TARGET_TICKER} — Simulated Price Distribution After {N_DAYS} Trading Days',
    xaxis_title='Price ($)', yaxis_title='Frequency'
)
fig_dist.show()

## Part 4 — Step-Size Sensitivity (Euler Accuracy vs $\Delta t$)

A key property of the Euler method is that **smaller $\Delta t$ gives higher accuracy**. Here we compare the approximation error for different step sizes — from daily ($\Delta t = 1/252$) to monthly ($\Delta t = 21/252$).

In [ ]:
step_configs = [
    (1,  'Daily (dt=1/252)'),
    (5,  'Weekly (dt=5/252)'),
    (21, 'Monthly (dt=21/252)')
]

print('Euler-Maruyama — Accuracy vs Step Size')
print('='*50)
print(f'  Theoretical mean (exact GBM): ${exact_mean:.4f}')
print()

errors = []
labels = []
np.random.seed(42)

for step_days, label in step_configs:
    dt_s     = step_days / 252.0
    mu_s     = mu    * step_days        # scale drift to step size
    sigma_s  = sigma * np.sqrt(step_days)  # scale vol to step size
    n_steps  = int(np.round(N_DAYS / step_days))

    paths_s       = np.zeros((n_steps + 1, 2000))
    paths_s[0]    = S0
    for t in range(1, n_steps + 1):
        Z           = np.random.standard_normal(2000)
        paths_s[t]  = (paths_s[t-1]
                       + mu_s   * paths_s[t-1] * dt_s
                       + sigma_s * paths_s[t-1] * np.sqrt(dt_s) * Z)

    sim_mean = paths_s[-1].mean()
    err      = abs(sim_mean - exact_mean) / exact_mean * 100
    errors.append(err)
    labels.append(label)
    print(f'  {label:<22}: simulated mean = ${sim_mean:.4f}  |  error = {err:.4f}%')

fig_err = go.Figure(go.Bar(
    x=labels, y=errors,
    marker_color=['#00e676', '#ffd740', '#ff5252'],
    text=[f'{e:.4f}%' for e in errors], textposition='outside'
))
fig_err.update_layout(
    template='plotly_dark', height=360,
    title='Euler-Maruyama Error vs Step Size  (smaller dt → higher accuracy)',
    xaxis_title='Step Size', yaxis_title='Mean Approximation Error (%)'
)
fig_err.show()

In [ ]:
# ── Save Results ─────────────────────────────────────────────────────────────
sim_summary = pd.DataFrame({
    'Day': np.arange(N_DAYS + 1),
    'P5':  p5,
    'P25': p25,
    'P50': p50,
    'P75': p75,
    'P95': p95
})
sim_summary.to_csv(f'../data/{TARGET_TICKER}_gbm_simulation.csv', index=False)

print(f'💾 Saved: {TARGET_TICKER}_gbm_simulation.csv')
print()
print('='*55)
print(f'✅ Notebook 6 complete! GBM Simulation Summary for {TARGET_TICKER}:')
print(f'  Starting Price     : ${S0:.2f}')
print(f'  Annual Drift  μ    : {mu_ann*100:+.2f}%')
print(f'  Annual Vol    σ    : {sigma_ann*100:.2f}%')
print(f'  1-Year Median Forecast : ${p50[-1]:.2f}')
print(f'  1-Year 90%% CI        : [${p5[-1]:.2f},  ${p95[-1]:.2f}]')
print()
print('All notebooks complete. Launch the dashboard: streamlit run dashboard/app.py')